# Customer Support Classification 

## Problem Understanding

This project looks at **short-text multi-class intent classification** in the customer support domain — given a short customer message, the model must predict which of 27 intents (for example *cancel_order*, *track_refund*, *contact_human_agent*) the customer is expressing.

The task is interesting for three reasons:

- **Short messages contain few useful words.** The median message is only about 9 words long, so there is very little text for the model to work with. Bag-of-words models rely on counting words, and with so few words per message there is not much to count — this is why contextual models, which also use word order and meaning, are more suitable here.
- **Some intents share the same vocabulary.** The 27 intents include pairs like *cancel_order* and *change_order* that use many of the same words. A model cannot separate them by keywords alone — it has to understand what the customer is actually asking for.
- **The data includes different writing styles.** The Bitext dataset labels each message with flags for things like misspellings, colloquial phrasing, and politeness. This means the data covers the kind of variation real customers produce, rather than only clean, well-formed sentences.


### What intent classification means

**Intent classification** = given a piece of text, predict which *intent* (purpose, goal) the writer is expressing, from a fixed list of possible intents.

Breaking that down:

- **Input:** a short piece of text (usually a customer message, a chatbot query, or a voice-assistant command).
- **Output:** one label from a predefined list.
- **The labels are intents** — they describe *what the user wants*, not *what the text is about*.

It is a specific sub-type of text classification. The word *intent* is what makes it intent classification rather than generic text classification.


### Research questions

- **RQ1 — Classical ceiling.** How well can a simple linear classifier over TF-IDF features separate 27 intents whose vocabulary overlaps within the same category?
- **RQ2 — Recurrent uplift.** Do sequence models (BiLSTM, GRU) with pretrained GloVe embeddings give a meaningful improvement over the classical baseline on messages this short?
- **RQ3 — Transformer payoff.** Do pretrained transformers (BERT-base, DistilBERT) justify their much larger size and training cost on a 27-class, 27k-sample dataset, and is DistilBERT close enough to BERT-base to be the deployable choice?
- **RQ4 — Failure structure.** Do classification errors concentrate inside semantically similar intent groups (as the EDA predicts), and does the error pattern change as models get more powerful?

### Dataset

The Bitext Customer Support LLM Chatbot Training Dataset is used — 26,872 messages, 27 intent classes grouped into 10 higher-level categories, loaded from HuggingFace. The dataset is synthetic: classes are nearly balanced (~995 messages per class), there are no missing values or duplicates, and each message carries linguistic-variation flags. The sample size comfortably exceeds the ≥6,000 minimum in the brief.

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available()) # Check if CUDA is available for GPU acceleration
print("Device name   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("VRAM (GB)     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
      if torch.cuda.is_available() else "n/a")

CUDA available: True
Device name   : NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM (GB)     : 12.8


c:\Users\Kasia\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5070 Ti Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 Ti Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [3]:
import torch
print(torch.cuda.is_available())          # True
#print(torch.cuda.get_device_name(0))      # Tesla P100-PCIE-16GB

import subprocess, sys
pkgs = ["torch","transformers","datasets","accelerate","evaluate",
        "tensorflow","scikit-learn","pandas","numpy","plotly",
        "requests","joblib","scipy"]
subprocess.check_call([sys.executable,"-m","pip","install","-q"]+pkgs)
print("All packages installed")

True
All packages installed


In [4]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, re, time, requests, io, joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from scipy.special import softmax
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [5]:
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

Defines section() (prints =×95 dividers with a title) and print_df() (styled pandas display helper).
Output: Helpers ready Keeps the output of every later cell visually consistent and easy to read.

In [6]:
# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def section(title: str): # function to print section titles in a consistent format
    print("\n" + "=" * 95)
    print(title)
    print("=" * 95)

def print_df(df: pd.DataFrame, name: str, n: int = 10): # function to print a dataframe's name, shape, and first n rows in a consistent format
    print("\n" + "-" * 95)
    print(f"[DATAFRAME] {name} | shape = {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    print("-" * 95)
    print(df.head(n).to_string())

COLORS_BINARY   = {"0": "#2E86AB", "1": "#D1495B"}
COLORS_SEQ      = px.colors.qualitative.Set2
COLORS_MULTI    = px.colors.qualitative.Set3 + px.colors.qualitative.Dark24
C_BLUE, C_RED   = "steelblue", "#D1495B"

print("Helpers ready ")

Helpers ready 


Config: Sets RANDOM_STATE=42, TARGET="intent", FEATURE_COL="instruction", NUM_CLASSES=27, DEVICE="cpu".
Output: Prints the config values and confirms PyTorch device is CPU, TensorFlow version.
Centralising these as constants means every model in the pipeline uses the same split seed, target column, and class count — a prerequisite for valid cross-model comparison.

In [7]:
# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
TARGET       = "intent"
FEATURE_COL  = "instruction"
NUM_CLASSES  = 27
DEVICE       = "cpu"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Target         : {TARGET}")
print(f"Feature        : {FEATURE_COL}")
print(f"PyTorch device : {DEVICE}")
print(f"TensorFlow     : {tf.__version__}")
print("Helpers ready ✓")

Target         : intent
Feature        : instruction
PyTorch device : cpu
TensorFlow     : 2.21.0
Helpers ready ✓


# Dataset

**Dataset:** Bitext Customer Support LLM Chatbot Training Dataset, loaded from HuggingFace (~27,000 samples).

**Target variable (Y):** `intent` — 27 distinct customer intent classes (e.g. cancel_order, get_refund, track_order). This is a multi-class classification problem with 27 possible output labels.

**Input feature (X):** `instruction` — a free-text customer utterance (e.g. "I need to cancel my order"). This is the only input feature used. No structured or numerical features are present in this dataset.

The intent label directly captures what the customer wants to achieve. Predicting it correctly allows a support system to route the customer to the right response or agent automatically.

The raw customer utterance contains all available signal. No metadata (e.g. account history, product type) is available — the model must classify intent from text alone.

**Task type:** Multi-class text classification — 27 mutually exclusive classes, one prediction per utterance.

Load Data: Calls the HuggingFace datasets-server API and downloads the Bitext parquet file directly into a pandas DataFrame, bypassing the datasets library (which has DLL issues on Windows with Application Control policies).
Output: Downloads the parquet file and prints a preview — 26,872 rows × 5 columns (flags, instruction, category, intent, response).

In [8]:
# =============================================================================
# LOAD DATA
# =============================================================================
section("Load dataset from HuggingFace")

def load_bitext() -> pd.DataFrame: # loading the Bitext dataset from HuggingFace's parquet API and returning it as a pandas DataFrame through API calls
    api = ("https://datasets-server.huggingface.co/parquet"
           "?dataset=bitext/Bitext-customer-support-llm-chatbot-training-dataset")
    resp = requests.get(api, timeout=30) # make an API request to get the parquet file URL for the Bitext dataset
    resp.raise_for_status() # check if the API request was successful
    url = resp.json()["parquet_files"][0]["url"] # extract the parquet file URL from the API response
    print(f"Downloading parquet: {url}")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

df = load_bitext()
print_df(df, "Bitext raw")


Load dataset from HuggingFace

-----------------------------------------------------------------------------------------------
[DATAFRAME] Bitext raw | shape = 26,872 rows × 5 cols
-----------------------------------------------------------------------------------------------
   flags                                                   instruction category        intent                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

# Data Preparation

Before any modelling, the raw text is checked, cleaned, and split into training, validation, and test sets.

This step does four things:

- **Inspect the data.** Shape, column types, missing values, and duplicates are checked so later problems are not caused by bad input.
- **Explore the data.** Class distribution, category grouping, message length, linguistic-variation flags, and sample messages are examined to understand what the models will have to deal with.
- **Clean the text.** Messages are lowercased, URLs and special characters are removed, and extra whitespace is stripped. This reduces vocabulary size without losing the words that actually carry intent information.
- **Split the data.** A stratified 70 / 15 / 15 split is used: 70% for training, 15% for validation (hyperparameter tuning, early stopping), and 15% for final test evaluation. Stratification guarantees all 27 classes are proportionally represented in every split.

The test set is only used once, at the end of the project, to produce the final reported metrics for each model.

# Exploration of Data

Before choosing a model, it helps to understand what the data actually looks like. The EDA checks the shape of the class distribution, the length of the messages, the vocabulary overlap between intents, and the type of linguistic variation the models will have to handle. Each finding below has a direct consequence for which models are suitable and how they should be evaluated.

- **Class distribution.** All 27 intents contain between 950 and 1,000 messages (~995 on average, ~3.7% of the dataset each). The data is near-perfectly balanced because it is synthetically generated. This means no class weighting or resampling is needed, and macro F1 is the correct main metric because every class contributes equally.
- **Category grouping.** The 27 intents belong to 10 higher-level categories (ACCOUNT, ORDER, REFUND, and so on). Some categories contain several intents that share vocabulary — ACCOUNT has 6 intents, ORDER has 4, REFUND has 3 — and these are the most likely sources of confusion in the confusion matrix later.
- **Message length.** Messages are short: median ~9 words, mean ~9 words, maximum 16. Short text weakens bag-of-words models because there are fewer discriminating words per message. This is part of the reason deeper contextual models are expected to help.
- **Linguistic variation flags.** Each message carries flags indicating its writing style — *Basic*, *Colloquial*, *Misspelled*, *Politeness*, *Question-like*, *Offensive*. Most messages are Basic, about a third are Question-like, and roughly 18% are Misspelled. The misspelled subset is the most important for robustness: a model that cannot handle typos is not suitable for real customer traffic, where spelling errors are common.
- **Sample messages.** Reading actual examples shows that similar intents share surface vocabulary — *cancel_order* and *change_order* both contain the words *order* and an action verb, and use `{{Order Number}}` template placeholders. This is qualitative evidence for the confusion hypothesis: TF-IDF alone cannot separate these intents, which is what motivates moving to contextual models.
- **Top words per intent.** Looking at the most common words for each intent shows how much vocabulary is shared between related intents. Intents with unique top words (e.g. *create_account*, *get_invoice*) are expected to score high F1 across all models. Intents whose top words overlap with neighbours (e.g. *cancel_order* vs *change_order*) are expected to produce the low-F1 tail.

These findings set the expectations for the modelling section: the data is clean and balanced, the main difficulty is vocabulary overlap between semantically close intents, and the most interesting robustness question is how each model tier handles misspelled messages.

### 1. Dataset Overview
**Checking for:** Shape, column types, missing values, duplicates.

**Findings:** 4 columns — instruction (input), intent (target), category (grouping), flags (linguistic variation). No missing values, no duplicates.

Prints shape, column names, missing-value counts per column, and duplicate row counts.
Output: Confirms 26,872 rows × 5 cols, 0 missing values, 0 duplicates, all columns are object (string) dtype.
Data is clean — no imputation or deduplication is needed. This is expected because Bitext is a synthetically generated dataset.

In [9]:
# =============================================================================
# DATASET OVERVIEW
# =============================================================================
section("Dataset overview")

print(f"Shape      : {df.shape[0]:,} rows × {df.shape[1]:,} cols")
print(f"Columns    : {list(df.columns)}")
print(f"Missing    : {df.isnull().sum().to_dict()}")
print(f"Duplicates : {df.duplicated().sum()}")
print()
print(df.dtypes.to_string())


Dataset overview
Shape      : 26,872 rows × 5 cols
Columns    : ['flags', 'instruction', 'category', 'intent', 'response']
Missing    : {'flags': 0, 'instruction': 0, 'category': 0, 'intent': 0, 'response': 0}
Duplicates : 0

flags          object
instruction    object
category       object
intent         object
response       object


### 2. Class Distribution
**Checking for:** Balance across all 27 intent classes.

**Findings:** Dataset is synthetically generated — near-perfect balance of ~1,000 samples per class. No class weighting or resampling required. Macro F1 is the appropriate primary metric as it treats all 27 classes equally.

Counts rows per intent, prints them as a table (count + percentage), and plots a Plotly bar chart.
Output: All 27 intents have between 950 and 1,000 samples (mean 995.3). Each class sits at ~3.7% of the dataset.
Near-perfect synthetic balance means no resampling or class weighting is needed. Macro F1 is the correct primary metric because it treats all 27 classes equally. ROC-AUC macro-OvR is a valid secondary metric under balance.

In [10]:
# =============================================================================
# TARGET DISTRIBUTION — 27 INTENT CLASSES
# =============================================================================
section("Target distribution (intent classes)")

vc = df[TARGET].value_counts().reset_index() # count the occurrences of each unique value in the target column and reset the index to get a DataFrame 
                                             # with 'intent' and 'count' columns
vc.columns = ["intent", "count"]
vc["pct"] = (vc["count"] / len(df) * 100).round(2)

print(vc.to_string(index=False))
print(f"\nUnique intents : {df[TARGET].nunique()}")
print(f"Min per class  : {vc['count'].min()}")
print(f"Max per class  : {vc['count'].max()}")
print(f"Mean per class : {vc['count'].mean():.1f}")

fig = px.bar( # plotting a horizontal bar chart of the intent class distribution using Plotly Express, with bars colored by count and a blue color scale
    vc.sort_values("count"),
    x="count", y="intent",
    orientation="h",
    title="Intent class distribution — all 27 classes",
    color="count",
    color_continuous_scale="Blues",
    labels={"count": "Number of samples", "intent": "Intent"}
)
fig.update_layout(title_x=0.02, height=700, yaxis_title=None)
fig.show()


Target distribution (intent classes)
                  intent  count  pct
contact_customer_service   1000 3.72
               complaint   1000 3.72
           check_invoice   1000 3.72
          switch_account   1000 3.72
            edit_account   1000 3.72
     contact_human_agent    999 3.72
   check_payment_methods    999 3.72
         delivery_period    999 3.72
 newsletter_subscription    999 3.72
             get_invoice    999 3.72
           payment_issue    999 3.72
   registration_problems    999 3.72
            cancel_order    998 3.71
             place_order    998 3.71
            track_refund    998 3.71
            change_order    997 3.71
 set_up_shipping_address    997 3.71
     check_refund_policy    997 3.71
          create_account    997 3.71
              get_refund    997 3.71
                  review    997 3.71
        delivery_options    995 3.70
          delete_account    995 3.70
        recover_password    995 3.70
             track_order    995 3.70


### 3. Category Distribution
**Checking for:** How the 27 intents group into 10 high-level categories.

**Model signals:** Categories with multiple similar intents (e.g. ORDER contains cancel_order and change_order) will produce the most confusion — confirmed in the confusion matrix later.

Groups the 27 intents into their high-level category column (ACCOUNT, ORDER, REFUND, etc.) and prints counts.
11 categories: ACCOUNT (5,986) , CANCEL (950). Heavy skew at the category level because ACCOUNT contains 6 intents.

In [11]:
# =============================================================================
# CATEGORY DISTRIBUTION (10 HIGH-LEVEL GROUPS)
# =============================================================================
section("Category distribution")

cat_vc = df["category"].value_counts().reset_index() # count the occurrences of each unique value in the 'category' column and reset the index to get a DataFrame with 'category' and 'count' columns
cat_vc.columns = ["category", "count"]

fig = px.bar(
    cat_vc,
    x="category", y="count",
    title="Sample count by category (high-level groups)",
    color="category",
    color_discrete_sequence=COLORS_SEQ,
    labels={"count": "Count", "category": "Category"}
)
fig.update_layout(title_x=0.02, showlegend=False, xaxis_tickangle=-30)
fig.show()

print(cat_vc.to_string(index=False))


Category distribution


    category  count
     ACCOUNT   5986
       ORDER   3988
      REFUND   2992
     CONTACT   1999
     INVOICE   1999
     PAYMENT   1998
    FEEDBACK   1997
    DELIVERY   1994
    SHIPPING   1970
SUBSCRIPTION    999
      CANCEL    950


### 4. Text Length Distribution
**Checking for:** Utterance length in words and characters.

**Model signals:** Median ~8 words. Short text weakens bag-of-words models as there are fewer discriminating terms per sample. Transformer models handle short sequences natively without truncation.

Computes n_chars and n_words per utterance, prints summary statistics, and plots the distribution.
Output: Mean 8.69 words / 46.89 chars, median 9, max 16. Very short utterances.
Why it matters: With a median of just 9 words, bag-of-words models have few discriminating terms per sample — this caps what TF-IDF can achieve. Transformer models handle short sequences natively, so this favours the transformer tier over classical ML.

In [12]:
# =============================================================================
# TEXT LENGTH ANALYSIS
# =============================================================================
section("Text length analysis")

df["n_chars"] = df[FEATURE_COL].str.len() # # checking the length of the text in characters by applying the string length function to the feature column and storing it in a new column 'n_chars'
df["n_words"] = df[FEATURE_COL].str.split().str.len() # checking the length of the text in words by splitting the text on whitespace and counting the resulting list's length, storing it in a new column 'n_words'

print("-" * 80)
print(df[["n_chars", "n_words"]].describe().round(2).to_string())

# creating histograms to visualise the distribution of character lengths and word counts in the instruction texts, and a box plot to show the word count distribution by category
fig = px.histogram(
    df, x="n_chars", nbins=60,
    title="Instruction — character length distribution",
    labels={"n_chars": "Character length", "count": "Count"}
)
fig.update_traces(marker_color=C_BLUE)
fig.update_layout(bargap=0.05, title_x=0.02)
fig.show()

fig = px.histogram(
    df, x="n_words", nbins=40,
    title="Instruction — word count distribution",
    labels={"n_words": "Word count", "count": "Count"}
)
fig.update_traces(marker_color=C_RED)
fig.update_layout(bargap=0.05, title_x=0.02)
fig.show()

fig = px.box( # creating a box plot to show the distribution of word counts by category, with boxes colored by category and using a predefined color sequence
    df, y="n_words", x="category",
    title="Word count by category",
    color="category",
    color_discrete_sequence=COLORS_SEQ
)
fig.update_layout(title_x=0.02, xaxis_tickangle=-30, showlegend=False)
fig.show()


Text length analysis
--------------------------------------------------------------------------------
        n_chars   n_words
count  26872.00  26872.00
mean      46.89      8.69
std       10.90      2.61
min        6.00      1.00
25%       40.00      7.00
50%       48.00      9.00
75%       55.00     11.00
max       92.00     16.00


### Intent density per category

This analysis counts how many distinct intents live inside each high-level category.

**Checking for:** Which categories contain the most intents — i.e. which categories will be most competitive for the model to separate.

**Findings:** ACCOUNT holds 6 intents, ORDER holds 4, REFUND holds 3 — all others have 1–2. These three crowded categories are the predicted confusion hotspots, because intents in the same category share vocabulary and action verbs (e.g. ORDER contains *cancel_order*, *change_order*, *delivery_options*, *track_order* — all share the word "order" plus an action verb).

**Model signals:** The confusion matrix later should show that most off-diagonal errors concentrate inside ACCOUNT, ORDER, and REFUND, and that cross-category errors are rare.

In [13]:
# =============================================================================
# INTENT DENSITY BY CATEGORY
# Count how many distinct intents live inside each high-level category.
# Categories with many intents are the predicted confusion hotspots because
# intents in the same category share vocabulary (e.g. ORDER contains
# cancel_order, change_order, delivery_options, track_order — all of which
# share the word "order" and an action verb).
# =============================================================================
section("Intent density per category")

intent_cat = (
    df.groupby("category")["intent"]
      .nunique()
      .reset_index()
      .rename(columns={"intent": "unique_intents"})
      .sort_values("unique_intents", ascending=False)
)

styled_table = (
    intent_cat.style
    .background_gradient(subset=["unique_intents"], cmap="Blues")
    .set_properties(**{
        "background-color": "white",
        "color": "black",
        "border-color": "black"
    })
)
styled_table


Intent density per category


,category,unique_intents
0,ACCOUNT,6
6,ORDER,4
8,REFUND,3
5,INVOICE,2
2,CONTACT,2
4,FEEDBACK,2
3,DELIVERY,2
9,SHIPPING,2
7,PAYMENT,2
1,CANCEL,1


### 6. Sample Utterances
**Checking for:** Vocabulary overlap between similar intents.

**Key finding:** cancel_order and change_order share keywords such as order, cancel, change. This overlap is the primary source of confusion matrix off-diagonal entries and appears across all model tiers.

Prints 3 raw example utterances each from a set of chosen intents (cancel_order, track_order, etc.).
Output: Examples showing {{Order Number}} template placeholders and real vocabulary overlap — for example, both cancel_order and change_order contain the words "order" and action verbs.
This is qualitative evidence for the confusion hypothesis — intents in the same category genuinely share keywords, so TF-IDF alone cannot separate them. This motivates moving beyond classical ML to contextual models.

In [14]:
# =============================================================================
# SAMPLE UTTERANCES PER INTENT
# =============================================================================
section("Sample utterances per intent (3 examples each)")

sample_intents = ["cancel_order", "track_order", "get_refund",
                  "contact_human_agent", "complaint"]
for intent_name in sample_intents: # function to print 3 random sample utterances for each of the specified intent names, 
                                    # with a header showing the intent name in uppercase and a bullet point for each sample
    print("\n" + "-" * 80)
    print(f"{intent_name.upper()}")
    samples = df[df[TARGET] == intent_name][FEATURE_COL].sample(3, random_state=42).tolist()
    for s in samples:
        print(f"  • {s}")


Sample utterances per intent (3 examples each)

--------------------------------------------------------------------------------
CANCEL_ORDER
  • I cannot afford purchase {{Order Number}}
  • I have bought some item, I have to cancel order {{Order Number}}
  • I do not want this item, cancel order {{Order Number}}

--------------------------------------------------------------------------------
TRACK_ORDER
  • where to see the bloody ETA of the order {{Order Number}}
  • checking status of order {{Order Number}}
  • see purchase {{Order Number}} current status

--------------------------------------------------------------------------------
GET_REFUND
  • I paid {{Currency Symbol}}{{Refund Amount}} for this product, I need to get a rebate
  • I am trying to receive a compensation
  • i need assistance demanding compensations of my money

--------------------------------------------------------------------------------
CONTACT_HUMAN_AGENT
  • I can't speak iwth an operator
  • I want he

### 7. Top Words per Intent
**Checking for:** Whether each intent has a sufficiently distinct vocabulary for bag-of-words separation.

**Model signals:** Intents with unique top words achieve high F1 across all models. Intents sharing top words with neighbours produce low F1 — particularly for TF-IDF-based classical ML.

Uses collections.Counter to compute the most frequent words for the three largest intents.
Output: Empty output — only the section header prints. The table is computed but never displayed.
This is a rendering bug to fix — likely a missing print(...) or display(...) at the end of the cell. The analysis is intended to show whether each intent has a discriminative top-N vocabulary.

In [ ]:
# =============================================================================
# TOP WORDS PER INTENT — TERM FREQUENCY OVERVIEW
# =============================================================================
section("Most common words per intent (top 3 intents by size)")

from collections import Counter

# For the top 3 intents by sample size, this code extracts the instruction texts, 
# tokenizes them into words, removes common stopwords and short words, counts the frequency of each remaining word, 
# and visualizes the top 10 most common words for each intent using horizontal bar charts colored by frequency.

top_intents = df[TARGET].value_counts().head(3).index.tolist()
for intent_name in top_intents:
    subset = df[df[TARGET] == intent_name][FEATURE_COL]
    words = " ".join(subset).lower().split()
    stopwords = {"i", "my", "to", "the", "a", "an", "is", "it", "me", "can", "you"}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    common = Counter(words).most_common(10)
    common_df = pd.DataFrame(common, columns=["word", "freq"])

    fig = px.bar(
        common_df, x="freq", y="word", orientation="h",
        title=f"Top words — {intent_name}",
        color="freq",
        color_continuous_scale="Blues",
        labels={"freq": "Frequency", "word": "Word"}
    )
    fig.update_layout(title_x=0.02, yaxis={"categoryorder": "total ascending"},
                      showlegend=False, height=350)
    fig.show()


Most common words per intent (top 3 intents by size)


## Feature Engineering & Representation

Text cannot be fed directly into any model — it must first be converted to numbers.
Three different representations are built for the two model tiers.

- **TF-IDF (for classical ML).** Each message is turned into a sparse numerical vector that records how informative each word (or two-word phrase) is for that message relative to the rest of the corpus. Unigrams and bigrams are used, the vocabulary is capped at 50,000 features, and the vectoriser is fitted on the training set only to avoid leakage.
- **Padded integer sequences + GloVe embeddings (for BiLSTM and GRU).** Each message is tokenised into integer IDs and padded to a fixed length of 50 tokens. The embedding layer is initialised with pretrained 100-dimensional GloVe vectors so the model starts with general word meaning learned from 6 billion words of text, rather than having to learn it from scratch on the small training set.
- **WordPiece tokenisation (for DistilBERT and BERT).** The transformer's own pretrained tokeniser is used, which splits rare words into known sub-word pieces (for example *cancellations* becomes `cancel` + `##lations`). This must be the checkpoint's own tokeniser — using any other would break the pretrained embedding layer.

Labels are encoded to integers 0–26 with `LabelEncoder`, fitted on the full label set so the mapping is the same across every split and every model.

### Text Cleaning

**Why clean text before modelling?** Raw customer utterances contain noise, mixed case, punctuation, URLs — that adds vocabulary size without adding discriminative signal. Reducing noise improves model generalisation.

**Decisions made and why:**

| Decision | Justification |
|----------|---------------|
| Lowercase | "Cancel" and "cancel" are the same word. Lowercasing reduces vocabulary size without losing meaning |
| Remove URLs | No URLs appear in customer utterances but included as a defensive step |
| Remove special characters | Punctuation (!, ?, .) adds noise without adding intent signal for classification |
| Normalise whitespace | Prevents tokenisation errors from extra spaces or newlines |
| Stemming / lemmatisation | TF-IDF with bigrams preserves enough morphological information. Stemming risks collapsing intent-relevant distinctions — e.g. "ordering" vs "ordered" |
| Stop word removal | Short utterances have few words — removing stop words could discard useful context (e.g. "I can't" vs "I can") |

In [ ]:
# ============================================================================= 
# TEXT CLEANING
# =============================================================================
section("Text cleaning")

def clean_text(t): # function to clean text by converting it to lowercase, removing URLs, special characters, and extra whitespace
    t = str(t).lower()
    t = re.sub(r"http\S+", "", t)          # remove URLs
    t = re.sub(r"[^a-z0-9\s]", " ", t)    # remove special chars
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text_clean"] = df[FEATURE_COL].apply(clean_text)

# Show before / after
print("-" * 80)
for i in range(3):
    print(f"RAW  : {df[FEATURE_COL].iloc[i]}")
    print(f"CLEAN: {df['text_clean'].iloc[i]}")
    print()


Text cleaning
--------------------------------------------------------------------------------
RAW  : question about cancelling order {{Order Number}}
CLEAN: question about cancelling order order number

RAW  : i have a question about cancelling oorder {{Order Number}}
CLEAN: i have a question about cancelling oorder order number

RAW  : i need help cancelling puchase {{Order Number}}
CLEAN: i need help cancelling puchase order number



### 9. Label Encoding

**Why:** Models require numerical labels, not strings. The intent column contains 27 text class names converted to integers 0–26.

**Method:** LabelEncoder — alphabetical ordering. Same encoder object reused across all notebooks for consistent label-to-intent mapping.

**Leakage check** — fitted on full label set (all 27 classes known upfront), no data leakage risk.


In [ ]:
# =============================================================================
# LABEL ENCODING
# =============================================================================
section("Label encoding")

from sklearn.preprocessing import LabelEncoder
# using label encoding to convert the target variable (intent) into numerical labels 
# that can be used for machine learning models, and storing the number of unique classes in NUM_CLASSES

le = LabelEncoder()
df["label"] = le.fit_transform(df[TARGET])
NUM_CLASSES = len(le.classes_)

print(f"Classes : {NUM_CLASSES}")
print("\nIntent → Label mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i:2d} → {cls}")


Label encoding
Classes : 27

Intent → Label mapping:
   0 → cancel_order
   1 → change_order
   2 → change_shipping_address
   3 → check_cancellation_fee
   4 → check_invoice
   5 → check_payment_methods
   6 → check_refund_policy
   7 → complaint
   8 → contact_customer_service
   9 → contact_human_agent
  10 → create_account
  11 → delete_account
  12 → delivery_options
  13 → delivery_period
  14 → edit_account
  15 → get_invoice
  16 → get_refund
  17 → newsletter_subscription
  18 → payment_issue
  19 → place_order
  20 → recover_password
  21 → registration_problems
  22 → review
  23 → set_up_shipping_address
  24 → switch_account
  25 → track_order
  26 → track_refund


### 10. Train / Validation / Test Split — 70 / 15 / 15

**split the data** To evaluate whether a model generalises to unseen examples. Training and evaluating on the same data produces misleadingly high metrics — the model may have simply memorised the training examples.

**Split rationale:**

| Split | Size | Purpose |
|-------|------|---------|
| Train (70%) | ~18,900 samples | Model training — ~700 examples per class |
| Validation (15%) | ~4,050 samples | Hyperparameter tuning and early stopping — never used for final metrics |
| Test (15%) | ~4,050 samples | Final evaluation — used exactly once per model, never during training |

**stratified:** With 27 classes, a random split risks some splits receiving zero samples for a particular intent. Stratification guarantees all 27 classes are proportionally represented in every split — essential for reliable evaluation.

**70/15/15:** 70% training provides ~700 samples per class — sufficient for classical ML and marginal for deep learning. Splitting the remaining 30% equally between validation and test gives balanced tuning and evaluation sets.

In [ ]:
# =============================================================================
# STRATIFIED TRAIN / VALIDATION / TEST SPLIT  70 / 15 / 15
# =============================================================================
section("Train / val / test split")

from sklearn.model_selection import train_test_split

# splitting the dataset into training, validation, and test sets using stratified sampling 
# to maintain the same class distribution across all sets, with 70% of the data for training, 15% for validation, and 15% for testing

X = df["text_clean"].values
y = df["label"].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=RANDOM_STATE)

print(f"Train : {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):,} samples  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):,} samples  ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total : {len(X):,}")
print(f"\nClasses in train: {len(np.unique(y_train))} | val: {len(np.unique(y_val))} | test: {len(np.unique(y_test))}")


Train / val / test split
Train : 18,809 samples (70.0%)
Val   : 4,032 samples  (15.0%)
Test  : 4,031 samples  (15.0%)
Total : 26,872

Classes in train: 27 | val: 27 | test: 27


---
### Feature Engineering — Deep Learning (BiLSTM, GRU)

Deep learning models (BiLSTM, GRU) require text converted to padded integer sequences fed through an embedding layer — not raw text strings.

**Why pretrained GloVe instead of random embeddings?**
With only ~700 training samples per class, the model does not have enough data to learn word meanings from scratch. GloVe vectors were pretrained on 6 billion words — they already know that "cancel" and "cancellation" are related before training even starts. The notebook will still run without GloVe (falls back to random initialisation) but results will be weaker.

**Two conditions tested for BiLSTM:**
- Frozen — GloVe weights fixed, pretrained knowledge preserved
- Fine-tuned — weights updated during training, adapts to customer support domain

Fits a Keras Tokeniser on the training text only, converts all three splits to integer sequences, and pads them to MAX_LEN=50.
Output: Vocabulary size 2,368 words, X_train_seq shape (18809, 50). max_len=50 chosen to cover the 95th percentile of utterance lengths from EDA.

Fitting the tokeniser on training data only is the correct leakage-free approach. The small vocabulary (2,368) reflects the narrow customer-support domain — useful because the model has less to learn, but also limits generalisation to out-of-domain text.

In [20]:
# =============================================================================
# FEATURE ENGINEERING — Deep Learning: BiLSTM, GRU
# Tokenisation + padding + GloVe embedding matrix
# =============================================================================
section("Tokenisation & padding Deep Learning")

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 30_000
MAX_LEN   = 50
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)  # fit on train only

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val),
                             maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding="post", truncating="post")

print(f"Vocabulary size  : {len(tokenizer.word_index):,}")
print(f"X_train_seq shape: {X_train_seq.shape}")
print(f"max_len=50 covers 95th percentile of utterance lengths from EDA")
print("\n Used by: NB3 (BiLSTM, GRU)")


Tokenisation & padding Deep Learning
Vocabulary size  : 2,368
X_train_seq shape: (18809, 50)
max_len=50 covers 95th percentile of utterance lengths from EDA

 Used by: NB3 (BiLSTM, GRU)


### Why GloVe Embeddings?

BiLSTM and GRU cannot read text — they need numbers. Each word is first
converted to an integer index (e.g. "cancel" → 452). The embedding layer
then maps that index to a dense vector of 100 numbers that the model can
learn from.

The simplest approach would be to start with random vectors and let the
model figure out word meanings during training. The problem is that with
only ~700 training samples per intent class, there is not enough data to
learn what words mean from scratch — the model would not converge well.

**GloVe provides pretrained vectors** where similar words are already close
together in vector space, learned from 6 billion words (Wikipedia +
Gigaword) before training even starts:

- "cancel" and "cancellation" → similar vectors 
- "refund" and "money back" → similar vectors 
- "cancel" and "invoice" → very different vectors 

This gives the model a semantic head start instead of starting from zero.

**GloVe is not strictly required** — the notebook will still run without it.
If the GloVe file is not downloaded, the code automatically falls back to
random initialisation and BiLSTM and GRU will still train. Results will
be a few percentage points lower but nothing will break.

**Two conditions are tested:**

| Condition | Embedding weights | What it tests |
|-----------|------------------|---------------|
| Frozen | Fixed — pretrained GloVe knowledge preserved | Is pretrained knowledge alone sufficient? |
| Fine-tuned | Updated during training | Does adapting GloVe to customer support language improve F1? |

Fine-tuned is expected to outperform frozen — GloVe was trained on
Wikipedia, not customer support conversations. Allowing the weights to
adjust gives the model a head start AND domain adaptation.

In [21]:
# =============================================================================
# GLOVE EMBEDDING MATRIX 
# Download: wget http://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip
# =============================================================================

# can be used for BiLSTM and GRU

section("GloVe embedding matrix — BiLSTM / GRU")

GLOVE_PATH = r"F:\Apps\chatbot_llm\glove.6B.100d.txt"
embeddings_index = {}
try:
    with open(GLOVE_PATH, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            embeddings_index[values[0]] = np.asarray(values[1:], dtype="float32")
    print(f"GloVe vectors loaded: {len(embeddings_index):,}")
except FileNotFoundError:
    print("GloVe not found — using random initialisation as fallback")
    #print("    Download: wget http://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip")

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index)) + 1
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
found = 0
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            found += 1

oov = 1 - found / (vocab_size - 1)
print(f"Embedding matrix : {embedding_matrix.shape}")
print(f"OOV rate         : {oov:.2%}")
print("\nUsed by: BiLSTM, GRU (frozen and fine-tuned conditions)")


GloVe embedding matrix — BiLSTM / GRU
GloVe vectors loaded: 400,000
Embedding matrix : (2369, 100)
OOV rate         : 65.37%

Used by: BiLSTM, GRU (frozen and fine-tuned conditions)


### Shared Evaluation Helper 

`get_metrics()` is defined here once so all three model notebooks use
the exact same function and return results in the same format.
This ensures NB5 can combine results from all three without any
manual editing.

Returns: `accuracy | precision | recall | f1 | roc_auc`

**Note for LinearSVC (NB2):** LinearSVC has no `predict_proba` — it uses
`decision_function` scores instead. These do not sum to 1 so they cannot
be passed directly to `roc_auc_score`. The function detects this
automatically and applies softmax to convert them to pseudo-probabilities
before computing ROC-AUC.

In [22]:
# =============================================================================
# SHARED EVALUATION HELPER — used by all four models
# =============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

def get_metrics(y_true, y_pred, y_proba,
                model_name=None, tier=None, time_s=None):
    """
    Shared evaluation function — works for all model types.
    model_name, tier, time_s are optional — include them when calling
    from BERT/GRU notebooks, skip them in classical ML notebooks.
    """
    scores = np.array(y_proba)
    if not np.allclose(scores.sum(axis=1), 1.0, atol=1e-3):
        scores = softmax(scores, axis=1)

    m = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1":        f1_score(y_true, y_pred, average="macro"),
        "roc_auc":   roc_auc_score(y_true, scores, multi_class="ovr", average="macro")
    }

    # Add optional fields if provided
    if model_name is not None: m["model"]  = model_name
    if tier       is not None: m["tier"]   = tier
    if time_s     is not None: m["time_s"] = round(time_s, 1)

    # Print results
    label = model_name if model_name else "Model"
    print(f"\n{label} — Test results:")
    for k,v in m.items():
        if k not in ["model","tier","time_s"]:
            print(f"  {k:12}: {v:.6f}")

    return m

def plot_history(history, model_name):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=history.history["loss"],     name="Train loss",
                             line=dict(color=C_BLUE)))
    fig.add_trace(go.Scatter(y=history.history["val_loss"], name="Val loss",
                             line=dict(color=C_RED, dash="dash")))
    fig.update_layout(title=f"{model_name} — Loss curves",
                      title_x=0.02, xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()

print("get_metrics() ready — accuracy | precision | recall | f1 | roc_auc")

get_metrics() ready — accuracy | precision | recall | f1 | roc_auc


---
### Feature Engineering for NB2 — Classical ML (Logistic Regression, LinearSVC)

**Representation: TF-IDF sparse matrix**

TF-IDF converts text to a numerical matrix where each value represents how informative a word is for a document relative to the corpus. Works directly with linear classifiers without dense embedding overhead.

| Parameter | Value | Why |
|-----------|-------|-----|
| ngram_range | (1,2) | Bigrams capture two-word phrases — cancel order, track refund |
| max_features | 50,000 | Caps vocabulary for memory efficiency |
| sublinear_tf | True | log(1+tf) — prevents high-frequency words dominating |
| min_df | 2 | Removes rare typos |

**Limitation:** Discards word order entirely. Motivates sequence models in NB3.

In [23]:
# =============================================================================
# FEATURE ENGINEERING — NB2 ONLY (Classical ML: LR, LinearSVC)
# TF-IDF sparse matrix — fitted on train, transform applied to val/test
# =============================================================================
section("TF-IDF vectorisation — for NB2 Classical ML")

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

X_train_tfidf = tfidf.fit_transform(X_train)  # fit + transform on train
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Vocabulary size  : {len(tfidf.vocabulary_):,} features")
print(f"X_train shape    : {X_train_tfidf.shape}")
print(f"Matrix sparsity  : {1 - X_train_tfidf.nnz/(X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.4f}")
print("\nUsed by: NB2 (Logistic Regression, LinearSVC)")


TF-IDF vectorisation — for NB2 Classical ML
Vocabulary size  : 4,901 features
X_train shape    : (18809, 4901)
Matrix sparsity  : 0.9972

Used by: NB2 (Logistic Regression, LinearSVC)


# Model Development

Three tiers of models are implemented so performance can be compared across fundamentally different architectures:

- **Classical ML (Tier 1).** Logistic Regression and LinearSVC are trained on the TF-IDF features. These establish the baseline — if a linear model over TF-IDF already performs well, the problem is mostly about lexical keywords and deeper models will only add marginal value.
- **Recurrent deep learning (Tier 2).** BiLSTM and GRU are trained on padded integer sequences with pretrained GloVe embeddings. Both models are tuned over six configurations each, testing model size, dropout, learning rate, and frozen vs fine-tuned embeddings. A BiLSTM reads the sentence in both directions, while a GRU is a lighter single-direction alternative.
- **Transformers (Tier 3).** DistilBERT and BERT-base-uncased are fine-tuned with their own WordPiece tokenisers. Four configurations are tested, varying the checkpoint, learning rate, and dropout. These models use self-attention, so every word in the message can look at every other word directly — a more expressive representation than the sequential hidden state of an RNN.

Each model is trained on the same training split, tuned on the same validation split, and evaluated on the same test split. Results are recorded through a single shared `get_metrics()` function so the comparison in Step 6 is fair.

### Model 1 — Logistic Regression

**Type:** Linear probabilistic classifier

**How it works:** Models the probability of each of the 27 intent classes directly. Finds a linear decision boundary in TF-IDF feature space that maximises the likelihood of the correct class labels.

**Why chosen:** Provides well-calibrated probability scores — useful for comparing confidence across classes. Good baseline for any text classification task.

**Assumption:** 

In [24]:
# =============================================================================
# MODEL A — LOGISTIC REGRESSION
# =============================================================================
section("Logistic Regression")

t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver="saga", n_jobs=-1)
cv_lr = cross_val_score(lr, X_train_tfidf, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
print(f"CV Macro F1 : {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")

lr.fit(X_train_tfidf, y_train)
y_pred_lr   = lr.predict(X_test_tfidf)
y_proba_lr  = lr.predict_proba(X_test_tfidf)

lr_metrics = get_metrics(y_test, y_pred_lr, y_proba_lr)
results["Logistic Regression"] = {"CV F1": cv_lr.mean(), "CV Std": cv_lr.std(),
                                   "Time(s)": round(time.time() - t0, 1), **lr_metrics}
print(pd.Series(lr_metrics).round(4).to_string())


Logistic Regression
CV Macro F1 : 0.9915 ± 0.0011

Model — Test results:
  accuracy    : 0.994046
  precision   : 0.994142
  recall      : 0.994062
  f1          : 0.994065
  roc_auc     : 0.999874
accuracy     0.9940
precision    0.9941
recall       0.9941
f1           0.9941
roc_auc      0.9999


### Model 2 — LinearSVC

**Type:** Linear maximum-margin classifier

Finds the hyperplane in TF-IDF feature space that maximises the margin between classes — the decision boundary is as far as possible from all training examples. Typically outperforms Logistic Regression on sparse high-dimensional text.

**Why chosen:** LinearSVC is the standard benchmark for text classification with TF-IDF. Expected to be the strongest classical model.

In [32]:
# =============================================================================
# MODEL B — LINEAR SVC
# =============================================================================
section("LinearSVC")

t0 = time.time()
svc = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)
cv_svc = cross_val_score(svc, X_train_tfidf, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
print(f"CV Macro F1 : {cv_svc.mean():.4f} ± {cv_svc.std():.4f}")

svc.fit(X_train_tfidf, y_train)
y_pred_svc  = svc.predict(X_test_tfidf)
# LinearSVC has no predict_proba — use decision_function scores for roc_auc (OvR)
y_proba_svc = svc.decision_function(X_test_tfidf)

svc_metrics = get_metrics(y_test, y_pred_svc, y_proba_svc)
results["LinearSVC"] = {"CV F1": cv_svc.mean(), "CV Std": cv_svc.std(),
                        "Time(s)": round(time.time() - t0, 1), **svc_metrics}
print(pd.Series(svc_metrics).round(4).to_string())


LinearSVC
CV Macro F1 : 0.9937 ± 0.0012

Model — Test results:
  accuracy    : 0.995038
  precision   : 0.995086
  recall      : 0.995050
  f1          : 0.995048
  roc_auc     : 0.999972
accuracy     0.9950
precision    0.9951
recall       0.9951
f1           0.9950
roc_auc      1.0000


## Step 5 — Evaluation & Results

**Checking for:** CV F1 ≈ Test F1 (within ~1%) — confirms good generalisation, no overfitting to training set. Sorted by ROC-AUC.

In [26]:
# =============================================================================
# FINAL MODEL COMPARISON
# =============================================================================
section("Classical ML — final comparison")

res_df = pd.DataFrame(results).T
# Column order matches df_raw: accuracy | precision | recall | f1 | roc_auc
res_df = res_df[["accuracy", "precision", "recall", "f1", "roc_auc", "CV F1", "CV Std", "Time(s)"]]
res_df = res_df.sort_values("roc_auc", ascending=False)

print("=== FINAL MODEL COMPARISON ===")
print(res_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].round(4).to_string())
res_df.to_csv("nb1_classical_ml_results.csv")

# summary table
styled_res = (
    res_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].style
    .background_gradient(subset=["roc_auc"],   cmap="Greens")
    .background_gradient(subset=["f1"],        cmap="Blues")
    .background_gradient(subset=["precision"], cmap="Oranges")
    .set_properties(**{"background-color": "white", "color": "black", "border-color": "black"})
    .format("{:.4f}")
)
styled_res


Classical ML — final comparison
=== FINAL MODEL COMPARISON ===
                     accuracy  precision  recall      f1  roc_auc
LinearSVC               0.995     0.9951  0.9951  0.9950   1.0000
Logistic Regression     0.994     0.9941  0.9941  0.9941   0.9999


,accuracy,precision,recall,f1,roc_auc
LinearSVC,0.9950,0.9951,0.9951,0.9950,1.0000
Logistic Regression,0.9940,0.9941,0.9941,0.9941,0.9999


In [27]:
# =============================================================================
# COMPARISON CHART — CV vs TEST MACRO F1
# =============================================================================
section("Comparison chart")

res_plot = res_df.reset_index().rename(columns={"index": "Model"})

fig = go.Figure()
fig.add_trace(go.Bar(
    name="CV Macro F1",
    x=res_plot["Model"], y=res_plot["CV F1"],
    marker_color="#2E86AB",
    error_y=dict(type="data", array=res_plot["CV Std"].tolist(), visible=True)
))
fig.add_trace(go.Bar(
    name="Test Macro F1",
    x=res_plot["Model"], y=res_plot["f1"],
    marker_color="#D1495B"
))
fig.update_layout(
    barmode="group",
    title="Classical ML — CV vs Test Macro F1",
    title_x=0.02,
    yaxis=dict(title="Macro F1", range=[0.5, 1.0]),
    legend_title_text="Split"
)
fig.show()


Comparison chart


In [28]:
# =============================================================================
# PER-INTENT F1 TABLE — LinearSVC
# =============================================================================
section("Per-intent F1 breakdown — LinearSVC")

import re as _re
report_dict = classification_report(y_test, y_pred_svc, target_names=le.classes_, output_dict=True)
intent_f1 = pd.DataFrame({
    "Intent":    le.classes_,
    "F1 Score":  [report_dict[c]["f1-score"] for c in le.classes_],
    "Precision": [report_dict[c]["precision"] for c in le.classes_],
    "Recall":    [report_dict[c]["recall"] for c in le.classes_],
    "Support":   [report_dict[c]["support"] for c in le.classes_],
}).sort_values("F1 Score")

fig = px.bar(
    intent_f1,
    x="F1 Score", y="Intent",
    orientation="h",
    title="Per-intent F1 score — LinearSVC",
    color="F1 Score",
    color_continuous_scale="RdYlGn",
    labels={"F1 Score": "F1 Score"}
)
fig.update_layout(title_x=0.02, height=700, yaxis_title=None)
fig.show()

print("\n5 hardest intents:")
print(intent_f1.head(5)[["Intent","F1 Score","Precision","Recall"]].round(4).to_string(index=False))
print("\n5 easiest intents:")
print(intent_f1.tail(5)[["Intent","F1 Score","Precision","Recall"]].round(4).to_string(index=False))


Per-intent F1 breakdown — LinearSVC



5 hardest intents:
        Intent  F1 Score  Precision  Recall
  change_order    0.9733     0.9669  0.9799
  cancel_order    0.9866     0.9932  0.9800
   track_order    0.9898     1.0000  0.9799
   get_invoice    0.9900     0.9933  0.9867
create_account    0.9900     0.9933  0.9867

5 easiest intents:
                 Intent  F1 Score  Precision  Recall
             get_refund       1.0        1.0     1.0
newsletter_subscription       1.0        1.0     1.0
                 review       1.0        1.0     1.0
       recover_password       1.0        1.0     1.0
           track_refund       1.0        1.0     1.0


### Confusion Matrix — LinearSVC

**Checking for:** Which intent pairs are most confused.

**Expected pattern:** Confusion concentrates between semantically similar intents in the same category — cancel_order vs change_order, track_order vs track_refund. These share vocabulary that TF-IDF cannot distinguish. Cross-category errors are rare.

In [29]:
# =============================================================================
# CONFUSION MATRIX — BEST MODEL (LinearSVC)
# =============================================================================
section("Confusion matrix — LinearSVC (best classical model)")

print(classification_report(y_test, y_pred_svc, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred_svc)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

fig = px.imshow(
    cm_df,
    title="Confusion matrix — LinearSVC (test set)",
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="True", color="Count"),
    aspect="auto",
    height=750
)
fig.update_layout(title_x=0.02)
fig.show()


Confusion matrix — LinearSVC (best classical model)
                          precision    recall  f1-score   support

            cancel_order       0.99      0.98      0.99       150
            change_order       0.97      0.98      0.97       149
 change_shipping_address       0.99      1.00      0.99       146
  check_cancellation_fee       1.00      1.00      1.00       142
           check_invoice       0.99      0.99      0.99       150
   check_payment_methods       1.00      1.00      1.00       150
     check_refund_policy       1.00      1.00      1.00       150
               complaint       1.00      1.00      1.00       150
contact_customer_service       1.00      0.99      1.00       150
     contact_human_agent       0.99      1.00      1.00       150
          create_account       0.99      0.99      0.99       150
          delete_account       0.98      1.00      0.99       149
        delivery_options       1.00      1.00      1.00       149
         delivery_peri

# << Mohamed Ibrahim - Updates >>

### Build the BiLSTM model

Below step is to define a reusable function that creates the BiLSTM classifier. The model uses an embedding layer, a bidirectional LSTM encoder, a pooling layer, and a dense classification head.

This structure is suitable for customer-support text because the meaning of a sentence often depends on both earlier and later words in the sequence.

In [30]:
'''
def has_glove_matrix():
    """Return True if the embedding matrix contains pretrained non-zero vectors."""
    return (
        "embedding_matrix" in globals()
        and embedding_matrix is not None
        and np.any(embedding_matrix != 0)
    )
print(f"GloVe available    : {has_glove_matrix()}")
print("BiLSTM setup ready ✓")
'''

'\ndef has_glove_matrix():\n    """Return True if the embedding matrix contains pretrained non-zero vectors."""\n    return (\n        "embedding_matrix" in globals()\n        and embedding_matrix is not None\n        and np.any(embedding_matrix != 0)\n    )\nprint(f"GloVe available    : {has_glove_matrix()}")\nprint("BiLSTM setup ready ✓")\n'

## Input summary

The BiLSTM receives padded token sequences as input and predicts one of the encoded intent classes.
The variables below were prepared earlier in the notebook and are reused here:

1. X_train_seq, X_val_seq, X_test_seq: padded token sequences
2. y_train, y_val, y_test: encoded target labels
3. MAX_LEN: maximum sequence length
4. EMBED_DIM: embedding size
5. NUM_CLASSES: number of intent classes

## Define the BiLSTM model

The next cell defines a reusable function that builds the BiLSTM classifier. The model includes:

1. an embedding layer
2. spatial dropout for regularisation
3. a bidirectional LSTM layer
4. global max pooling
5. a dense classification head
6. a softmax output layer

This structure is appropriate for short customer-support messages because the important intent cues may appear anywhere in the sequence.

## BiLSTM setup

A small helper function is also defined to check whether the GloVe embedding matrix is available. This is useful because the model should still work even if pretrained embeddings are missing.

Pretrained GloVe was chosen for three reasons.

**First**, the Bitext training set contains only ~18,800 utterances after splitting — far too few to learn useful embeddings from scratch for a 2,368-word vocabulary. Random initialisation would force the model to spend most of its capacity learning that *cancel* and *refund* are different words, rather than learning the intent-classification task itself.

**Second**, GloVe vectors already encode general-purpose lexical semantics learned from billions of co-occurrence statistics, so the model starts with a strong prior — synonyms cluster together, and morphological variants (*cancel / cancelled / cancelling*) point in similar directions. This gives faster convergence and better generalisation to unseen phrasings in the test set.

**Third**, the 100-dimensional variant was selected as a deliberate balance: large enough to carry meaningful semantic structure, small enough to keep the embedding matrix compact (~24k × 100 = 2.4M parameters) and avoid overfitting on a small training corpus.

---

The embedding matrix is built by looking up each tokeniser-vocabulary word in the GloVe dictionary; words not found (out-of-vocabulary) are left as zero vectors. The OOV rate is reported in the notebook as a sanity check, a high OOV rate would indicate that GloVe is a poor match for the customer-support domain and that domain-specific embeddings (e.g. fastText trained on support tickets) should be considered instead.

By default the embedding layer is **frozen** during training so the pretrained semantics are preserved; one tuning trial **unfreezes** the layer to allow domain adaptation, and the trade-off between the two is reported in the BiLSTM tuning table.

In [31]:
from tensorflow.keras import models

In [2]:
# =============================================================================
# BILSTM MODEL BUILDER - change to use glove only? 
# =============================================================================

section("Define BiLSTM model")

glove_available = GLOVE_PATH
# Sanity check — fail fast if GloVe didn't load, rather than silently using zeros
assert embedding_matrix is not None and np.any(embedding_matrix != 0), (
    "GloVe embedding matrix is empty — check GLOVE_PATH in the embedding cell."
)

def build_bilstm_model(
    lstm_units=128,
    dense_units=64,
    lstm_dropout=0.2,
    dense_dropout=0.4,
    spatial_dropout=0.2,
    learning_rate=1e-3,
    trainable_embeddings=False, # should we add trainable? True
    model_name="BiLSTM",
):
    """
    Build a BiLSTM text classifier using the existing notebook variables.
    """

    # If GloVe is unavailable, embeddings must be trainable
    glove_available = GLOVE_PATH
    #glove_available = has_glove_matrix()
    effective_trainable = trainable_embeddings if glove_available else True

    model = models.Sequential(name="BiLSTM_classifier")

    # Input = padded sequence of token ids
    model.add(layers.Input(shape=(MAX_LEN,), name="input_ids"))

    # Embedding layer
    if glove_available:
        model.add(
            layers.Embedding(
                input_dim=vocab_size,
                output_dim=EMBED_DIM,
                weights=[embedding_matrix],
                trainable=effective_trainable,
                mask_zero=True,
                name="embedding"
            )
        )
    else:
        model.add(
            layers.Embedding(
                input_dim=vocab_size,
                output_dim=EMBED_DIM,
                trainable=True,
                mask_zero=True,
                name="embedding"
            )
        )

    # Regularisation at the embedding level
    model.add(layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout"))

    # Bidirectional LSTM
    model.add(
        layers.Bidirectional(
            layers.LSTM(
                lstm_units,
                dropout=lstm_dropout,
                return_sequences=True
            ),
            name="bilstm"
        )
    )

    # Pool strongest signal across the sequence
    model.add(layers.GlobalMaxPooling1D(name="global_max_pool"))

    # Dense classification head
    model.add(layers.Dense(dense_units, activation="relu", name="dense_relu"))
    model.add(layers.Dropout(dense_dropout, name="dense_dropout"))

    # Final multi-class classifier
    model.add(layers.Dense(NUM_CLASSES, activation="softmax", name="classifier"))

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("build_bilstm_model() ready ✓")

NameError: name 'section' is not defined

### Why this BiLSTM architecture was chosen

This architecture was selected because it is *appropriate for multi-class text classification.*

The embedding layer converts token IDs into dense vector representations. If GloVe vectors are available, the model starts from pretrained semantic information. The parameter trainable_embeddings allows the experiment to compare frozen embeddings against fine-tuned embeddings.

**SpatialDropout1D** is used to improve generalisation by dropping parts of embedding features during training.

The Bidirectional LSTM reads the message in both directions. This is useful because important intent cues may appear at the beginning, middle, or end of a customer message.

**GlobalMaxPooling1D** reduces the sequence output into one fixed-size vector by keeping the strongest feature activations. This often works well for intent classification because a few important words or phrases can be highly informative.

The output layer uses softmax because this is a multi-class classification problem with NUM_CLASSES intent labels.

The loss function is sparse categorical cross-entropy because the target labels are already integer encoded rather than one-hot encoded.

### Training helper for BiLSTM experiments

The following function trains one BiLSTM configuration and returns:

1. The trained model
2. The training history
3. The training history
4. A validation summary used for hyperparameter tuning
the total training time

The validation metrics are computed directly inside the function so that the shared helper get_metrics() can remain the common final evaluation function for all models.

In [97]:
# =============================================================================
# TRAINING HELPER FOR BILSTM
# =============================================================================
section("BiLSTM training helper")

def train_one_bilstm(config, verbose=0):
    """
    Train one BiLSTM configuration and evaluate it on the validation set.
    Returns model, history, validation summary, and training time.
    """
    K.clear_session()
    gc.collect()

    model = build_bilstm_model(
        lstm_units=config["lstm_units"],
        dense_units=config["dense_units"],
        lstm_dropout=config["lstm_dropout"],
        dense_dropout=config["dense_dropout"],
        spatial_dropout=config["spatial_dropout"],
        learning_rate=config["learning_rate"],
        trainable_embeddings=config["trainable_embeddings"]
    )

    cbs = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-5,
            verbose=1
        )
    ]

    start = time.time()

    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=15,
        batch_size=config["batch_size"],
        callbacks=cbs,
        verbose=verbose
    )

    train_time = time.time() - start

    # Validation predictions for tuning
    val_proba = model.predict(X_val_seq, verbose=0)
    val_pred = val_proba.argmax(axis=1)

    val_summary = {
        "trial_name": config["trial_name"],
        "lstm_units": config["lstm_units"],
        "dense_units": config["dense_units"],
        "lstm_dropout": config["lstm_dropout"],
        "dense_dropout": config["dense_dropout"],
        "spatial_dropout": config["spatial_dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "trainable_embeddings": config["trainable_embeddings"] if glove_available else True,
        "epochs_ran": len(history.history["loss"]),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "train_time_s": round(train_time, 1),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall": recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1": f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc": roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
    }

    return model, history, val_summary, train_time

print("train_one_bilstm() ready ✓")


BiLSTM training helper
train_one_bilstm() ready ✓


### Why this training function is needed

This helper function makes the notebook easier to manage because it standardises how each BiLSTM configuration is trained and compared.

***EarlyStopping*** is used to stop training when validation loss stops improving, which helps reduce overfitting.

***ReduceLROnPlateau*** lowers the learning rate when progress slows down, which can improve optimisation.

The model is tuned using validation macro F1-score because this is a multi-class classification task, and macro F1 gives balanced importance to each intent class..

## Define the hyperparameter search space

The next cell defines a small but meaningful tuning space for the BiLSTM. The selected hyperparameters control the model capacity, regularisation strength, optimisation behaviour, and whether the pretrained embeddings are frozen or fine-tuned.

###Why these hyperparameters were selected

A hyperparameter search is used here to keep the tuning process efficient while still exploring the most important BiLSTM design choices:

1. smaller vs larger BiLSTM capacity
2. frozen vs trainable embeddings
3. moderate differences in dropout and learning rate

This is sufficient for the assignment and keeps the experiments interpretable.

In [98]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE
# =============================================================================
section("BiLSTM hyperparameter search space")

bilstm_search_space = [
    {
        "trial_name": "baseline_frozen",
        "lstm_units": 64,
        "dense_units": 64,
        "lstm_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "trainable_embeddings": False
    },
    {
        "trial_name": "frozen_larger",
        "lstm_units": 128,
        "dense_units": 64,
        "lstm_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "trainable_embeddings": False
    },
    {
        "trial_name": "frozen_higher_dropout",
        "lstm_units": 128,
        "dense_units": 64,
        "lstm_dropout": 0.3,
        "dense_dropout": 0.5,
        "spatial_dropout": 0.3,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "trainable_embeddings": False
    },
    {
        "trial_name": "finetune_small",
        "lstm_units": 64,
        "dense_units": 64,
        "lstm_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "trainable_embeddings": True
    },
    {
        "trial_name": "finetune_medium",
        "lstm_units": 128,
        "dense_units": 64,
        "lstm_dropout": 0.2,
        "dense_dropout": 0.4,
        "spatial_dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "trainable_embeddings": True
    },
    {
        "trial_name": "finetune_richer_head",
        "lstm_units": 128,
        "dense_units": 128,
        "lstm_dropout": 0.3,
        "dense_dropout": 0.5,
        "spatial_dropout": 0.3,
        "learning_rate": 3e-4,
        "batch_size": 32,
        "trainable_embeddings": True
    }
]

pd.DataFrame(bilstm_search_space)


BiLSTM hyperparameter search space


,trial_name,lstm_units,dense_units,lstm_dropout,dense_dropout,spatial_dropout,learning_rate,batch_size,trainable_embeddings
0,baseline_frozen,64,64,0.2,0.3,0.2,0.0010,64,False
1,frozen_larger,128,64,0.2,0.3,0.2,0.0010,64,False
2,frozen_higher_dropout,128,64,0.3,0.5,0.3,0.0005,64,False
3,finetune_small,64,64,0.2,0.3,0.2,0.0005,64,True
4,finetune_medium,128,64,0.2,0.4,0.2,0.0005,64,True
5,finetune_richer_head,128,128,0.3,0.5,0.3,0.0003,32,True


### Why these four trials were chosen

The first trial, baseline_frozen, is the reference model.
The second, frozen_larger, tests whether increasing the number of LSTM units improves performance while keeping embeddings fixed.
The third, finetune_small, tests whether allowing pretrained embeddings to update during training improves the baseline.
The fourth, finetune_medium, combines a larger recurrent layer with trainable embeddings and slightly stronger dropout.

Together, these four trials provide a clear and manageable tuning study.

## Run hyperparameter tuning

The following cell trains each BiLSTM configuration and compares them using validation performance. The best configuration is selected using validation macro F1-score, while validation loss is used as a tie-breaker.

The test set is not used during this stage.

In [94]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks, models
import tensorflow.keras.backend as K
import gc

In [101]:
# =============================================================================
# RUN HYPERPARAMETER TUNING
# =============================================================================
section("Run BiLSTM hyperparameter tuning")

tuning_rows = []
best_config = None
best_val_f1 = -1
best_val_loss = np.inf

for i, config in enumerate(bilstm_search_space, start=1):
    print("\n" + "-" * 95)
    print(f"Trial {i}/{len(bilstm_search_space)} : {config['trial_name']}")
    print(config)

    _, _, summary, _ = train_one_bilstm(config=config, verbose=0)
    tuning_rows.append(summary)

    print(f"Val accuracy : {summary['val_accuracy']:.4f}")
    print(f"Val F1       : {summary['val_f1']:.4f}")
    print(f"Val ROC-AUC  : {summary['val_roc_auc']:.4f}")
    print(f"Best val loss: {summary['best_val_loss']:.4f}")
    print(f"Epochs ran   : {summary['epochs_ran']}")

    if (summary["val_f1"] > best_val_f1) or (
        np.isclose(summary["val_f1"], best_val_f1) and summary["best_val_loss"] < best_val_loss
    ):
        best_val_f1 = summary["val_f1"]
        best_val_loss = summary["best_val_loss"]
        best_config = config.copy()

bilstm_tuning_df = (
    pd.DataFrame(tuning_rows)
    .sort_values(["val_f1", "val_accuracy", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

print("\nBest BiLSTM configuration:")
print(best_config)

print("\nValidation comparison table:")
print(
    bilstm_tuning_df[
        [
            "trial_name", "lstm_units", "dense_units", "trainable_embeddings",
            "batch_size", "learning_rate", "epochs_ran",
            "val_accuracy", "val_f1", "val_roc_auc", "best_val_loss"
        ]
    ].round(4).to_string(index=False)
)


Run BiLSTM hyperparameter tuning

-----------------------------------------------------------------------------------------------
Trial 1/6 : baseline_frozen
{'trial_name': 'baseline_frozen', 'lstm_units': 64, 'dense_units': 64, 'lstm_dropout': 0.2, 'dense_dropout': 0.3, 'spatial_dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'trainable_embeddings': False}

Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 14: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Restoring model weights from the end of the best epoch: 15.
Val accuracy : 0.9896
Val F1       : 0.9896
Val ROC-AUC  : 1.0000
Best val loss: 0.0266
Epochs ran   : 15

-----------------------------------------------------------------------------------------------
Trial 2/6 : frozen_larger
{'trial_name': 'frozen_larger', 'lstm_units': 128, 'dense_units': 64, 'lstm_dropout': 0.2, 'dense_dropout': 0.3, 'spatial_dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'traina

## Hyperparameter tuning results

Tuning results

This step identifies the best-performing BiLSTM configuration using validation performance only. This approach avoids test-set leakage and ensures that the final evaluation remains fair.

The main selection metric is macro F1-score, because it reflects balanced performance across all intent classes.

## Visual comparison of validation performance

A visual comparison makes it easier to show how each BiLSTM configuration performed during tuning.

In [102]:
# =============================================================================
# VISUALISE TUNING RESULTS
# =============================================================================
section("BiLSTM tuning comparison")

fig = px.bar(
    bilstm_tuning_df.sort_values("val_f1", ascending=True),
    x="val_f1",
    y="trial_name",
    orientation="h",
    color="trainable_embeddings",
    title="BiLSTM validation macro F1 by trial",
    labels={"val_f1": "Validation macro F1", "trial_name": "Trial"}
)
fig.update_layout(title_x=0.02, yaxis_title=None)
fig.show()


BiLSTM tuning comparison


## Compare the baseline and tuned BiLSTM

After tuning, the baseline BiLSTM and the best tuned BiLSTM are trained and compared directly. This helps show whether hyperparameter tuning resulted in a meaningful performance improvement.

In [103]:
# =============================================================================
# FINAL TRAINING — BASELINE VS TUNED
# =============================================================================
section("Final BiLSTM training: baseline vs tuned")

baseline_config = bilstm_search_space[0]

baseline_model, baseline_history, baseline_val_summary, baseline_time = train_one_bilstm(
    config=baseline_config,
    verbose=0
)

tuned_model, tuned_history, tuned_val_summary, tuned_time = train_one_bilstm(
    config=best_config,
    verbose=0
)

plot_history(baseline_history, "BiLSTM (baseline)")
plot_history(tuned_history, "BiLSTM (tuned)")


Final BiLSTM training: baseline vs tuned

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Restoring model weights from the end of the best epoch: 14.

Epoch 11: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Restoring model weights from the end of the best epoch: 14.


## Why the baseline comparison is important

The baseline model acts as a reference point. Without a baseline, it would be difficult to show whether the tuning process truly improved the BiLSTM.

The training curves also help assess whether the models converged properly and whether overfitting occurred.

## Final test evaluation using the shared evaluation helper

The held-out test set is used only once at the end of the tuning process. The existing shared get_metrics() helper is reused so that evaluation remains consistent across all models in the project.

In [104]:
# =============================================================================
# TEST SET EVALUATION — USE SHARED HELPER
# =============================================================================
section("BiLSTM test evaluation")

baseline_proba = baseline_model.predict(X_test_seq, verbose=0)
baseline_pred = baseline_proba.argmax(axis=1)

tuned_proba = tuned_model.predict(X_test_seq, verbose=0)
tuned_pred = tuned_proba.argmax(axis=1)

baseline_metrics = get_metrics(
    y_true=y_test,
    y_pred=baseline_pred,
    y_proba=baseline_proba,
    model_name="BiLSTM (baseline)",
    tier="Deep Learning",
    time_s=baseline_time
)

tuned_metrics = get_metrics(
    y_true=y_test,
    y_pred=tuned_pred,
    y_proba=tuned_proba,
    model_name="BiLSTM (tuned)",
    tier="Deep Learning",
    time_s=tuned_time
)

results["BiLSTM (baseline)"] = baseline_metrics
results["BiLSTM (tuned)"] = tuned_metrics

bilstm_test_compare = (
    pd.DataFrame([baseline_metrics, tuned_metrics])
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("\nTest comparison:")
print(
    bilstm_test_compare[
        ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "time_s"]
    ].round(4).to_string(index=False)
)


BiLSTM test evaluation

BiLSTM (baseline) — Test results:
  accuracy    : 0.991813
  precision   : 0.991968
  recall      : 0.991825
  f1          : 0.991825
  roc_auc     : 0.999987

BiLSTM (tuned) — Test results:
  accuracy    : 0.995287
  precision   : 0.995315
  recall      : 0.995297
  f1          : 0.995293
  roc_auc     : 0.999993

Test comparison:
            model  accuracy  precision  recall     f1  roc_auc  time_s
   BiLSTM (tuned)    0.9953     0.9953  0.9953 0.9953      1.0   273.4
BiLSTM (baseline)    0.9918     0.9920  0.9918 0.9918      1.0   146.7


## Final performance comparison

The final test comparison reports the main classification metrics:

1. accuracy
2. macro precision
3. macro recall
4. macro F1-score
5. ROC-AUC
6. training time

Among these, macro F1-score is the most important metric because it gives balanced importance to each intent class.

## Classification report for the tuned BiLSTM

A detailed classification report is generated for the tuned BiLSTM. This helps identify which intent classes are predicted well and which classes remain more difficult.

In [105]:
# =============================================================================
# CLASSIFICATION REPORT — TUNED MODEL
# =============================================================================
section("Classification report — BiLSTM (tuned)")

print(
    classification_report(
        y_test,
        tuned_pred,
        target_names=le.classes_,
        digits=4,
        zero_division=0
    )
)


Classification report — BiLSTM (tuned)
                          precision    recall  f1-score   support

            cancel_order     0.9933    0.9933    0.9933       150
            change_order     1.0000    0.9933    0.9966       149
 change_shipping_address     0.9864    0.9932    0.9898       146
  check_cancellation_fee     1.0000    1.0000    1.0000       142
           check_invoice     0.9865    0.9733    0.9799       150
   check_payment_methods     1.0000    1.0000    1.0000       150
     check_refund_policy     1.0000    1.0000    1.0000       150
               complaint     1.0000    1.0000    1.0000       150
contact_customer_service     1.0000    0.9867    0.9933       150
     contact_human_agent     0.9868    0.9933    0.9900       150
          create_account     0.9933    0.9933    0.9933       150
          delete_account     0.9933    1.0000    0.9967       149
        delivery_options     0.9933    1.0000    0.9967       149
         delivery_period     1.0000

## Confusion matrix for the tuned BiLSTM

The confusion matrix provides a visual summary of prediction errors and helps identify which intent classes are commonly confused with one another.

In [106]:
# =============================================================================
# CONFUSION MATRIX — TUNED MODEL
# =============================================================================
section("Confusion matrix — BiLSTM (tuned)")

cm = confusion_matrix(y_test, tuned_pred)

fig = px.imshow(
    cm,
    color_continuous_scale="Blues",
    aspect="auto",
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="BiLSTM (tuned) — confusion matrix"
)
fig.update_layout(title_x=0.02)
fig.show()


Confusion matrix — BiLSTM (tuned)


# **Model Development: Gated Recurrent Units (GRU)**

In [ ]:
section("GRU Model Definition")
#glove_available=len(embeddings_index) >0

'''
GRU is a lighter recurrent unit than LSTM — it has 2 gates instead of 3,
    so it trains faster with similar accuracy on short text. Used here as a
    direct comparison point against BiLSTM to show whether bidirectionality
    + extra gating is worth the extra parameters on this dataset.
'''

def build_gru_model(
    gru_units=128,
    dense_units=64,
    gru_dropout=0.2,
    dense_dropout=0.4,
    spatial_dropout=0.2,
    learning_rate=1e-3,
    trainable_embeddings=False,
    model_name="GRU",
):

#Build a GRU text classifier using the existing notebook variables.

  effective_trainable = trainable_embeddings if glove_available else True
  model_gru = models.Sequential(name="GRU_classifier")

#input layer
  model_gru.add(layers.Input(shape=(MAX_LEN,), name="input_ids"))
  #Embedding layer
  if glove_available:
    model_gru.add(
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=EMBED_DIM,
            weights=[embedding_matrix],
            trainable=effective_trainable,
            mask_zero=True,
            name="embedding"
        )
    )
  else:
    model_gru.add(
            layers.Embedding(
                input_dim=vocab_size,
                output_dim=EMBED_DIM,
                trainable=True,
                mask_zero=True,
                name="embedding"
            )
        )

#Embedding regularisation
  model_gru.add(layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout"))

#GRU encoder
  model_gru.add(
    layers.GRU(
        gru_units,
        dropout=gru_dropout,
        return_sequences=True,
        name="gru"
    ),
  )

#pooling
  model_gru.add(layers.GlobalMaxPooling1D(name="global_max_pool"))

#Dense head
  model_gru.add(layers.Dense(dense_units, activation="relu", name="dense_relu"))
  model_gru.add(layers.Dropout(dense_dropout, name="dense_dropout"))

#Output layer
  model_gru.add(layers.Dense(NUM_CLASSES, activation="softmax", name="classifier"))

  model_gru.compile(
    optimizer=optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
  )

  return model_gru

print("build_gru_model() ready ✓")


GRU Model Definition
build_gru_model() ready ✓


### Why this GRU architecture was chosen

The architecture deliberately mirrors the BiLSTM so the two models can be compared on equal footing — same embedding layer, same regularisation strategy, same dense head, same output layer. The only meaningful difference is the recurrent unit itself.

**GRU vs BiLSTM trade-off:**
- **GRU** has 2 gates (update + reset) and one hidden state. It trains faster and uses fewer parameters.
- **BiLSTM** has 3 gates (input + forget + output), separate cell state, and reads the sequence in both directions.

For short utterances (~8 words median) this dataset doesn't strongly require backwards context, so GRU is a reasonable lighter alternative. If GRU matches or beats BiLSTM here, it suggests bidirectionality and the extra LSTM gating add complexity without payoff at this sequence length.

In [ ]:
# =============================================================================
# TRAINING HELPER FOR GRU
# Mirrors train_one_bilstm() — same callbacks, same return signature
# =============================================================================
section("GRU training helper")

def train_one_gru(config, verbose=0):
    """Train one GRU configuration and return model, history, val summary, time."""
    K.clear_session()
    gc.collect()

    model = build_gru_model(
        gru_units=config["gru_units"],
        dense_units=config["dense_units"],
        gru_dropout=config["gru_dropout"],
        dense_dropout=config["dense_dropout"],
        spatial_dropout=config["spatial_dropout"],
        learning_rate=config["learning_rate"],
        trainable_embeddings=config["trainable_embeddings"],
        model_name=f"GRU_{config['trial_name']}",   # unique per trial
    )

    cbs = [
        # Stop when val_loss stops improving — prevents overfitting and saves time
        callbacks.EarlyStopping(
            monitor="val_loss", patience=3,
            restore_best_weights=True, verbose=1,
        ),
        # Halve LR when val_loss plateaus — helps converge to a better minimum
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=1, min_lr=1e-5, verbose=1,
        ),
    ]

    start = time.time()
    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=15,
        batch_size=config["batch_size"],
        callbacks=cbs,
        verbose=verbose,
    )
    train_time = time.time() - start

    # Validation predictions used for tuning — test set is held out until final eval
    val_proba = model.predict(X_val_seq, verbose=0)
    val_pred  = val_proba.argmax(axis=1)

    val_summary = {
        "trial_name":           config["trial_name"],
        "gru_units":            config["gru_units"],
        "dense_units":          config["dense_units"],
        "gru_dropout":          config["gru_dropout"],
        "dense_dropout":        config["dense_dropout"],
        "spatial_dropout":      config["spatial_dropout"],
        "learning_rate":        config["learning_rate"],
        "batch_size":           config["batch_size"],
        "trainable_embeddings": config["trainable_embeddings"],
        "epochs_ran":           len(history.history["loss"]),
        "best_val_loss":        float(np.min(history.history["val_loss"])),
        "train_time_s":         round(train_time, 1),
        "val_accuracy":         accuracy_score(y_val, val_pred),
        "val_precision":        precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall":           recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1":               f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc":          roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro"),
    }
    return model, history, val_summary, train_time

print("train_one_gru() ready")

In [ ]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE — GRU
# Mirrors the BiLSTM search space (frozen vs fine-tuned, small vs larger)
# so the two tiers are tuned with comparable budgets
# =============================================================================
section("GRU hyperparameter search space")

gru_search_space = [
    {
        "trial_name": "baseline_frozen",
        "gru_units": 64, "dense_units": 64,
        "gru_dropout": 0.2, "dense_dropout": 0.3, "spatial_dropout": 0.2,
        "learning_rate": 1e-3, "batch_size": 64,
        "trainable_embeddings": False,
    },
    {
        "trial_name": "frozen_larger",
        "gru_units": 128, "dense_units": 64,
        "gru_dropout": 0.2, "dense_dropout": 0.3, "spatial_dropout": 0.2,
        "learning_rate": 1e-3, "batch_size": 64,
        "trainable_embeddings": False,
    },
    {
        "trial_name": "frozen_higher_dropout",
        "gru_units": 128, "dense_units": 64,
        "gru_dropout": 0.3, "dense_dropout": 0.5, "spatial_dropout": 0.3,
        "learning_rate": 5e-4, "batch_size": 64,
        "trainable_embeddings": False,
    },
    {
        "trial_name": "finetune_small",
        "gru_units": 64, "dense_units": 64,
        "gru_dropout": 0.2, "dense_dropout": 0.3, "spatial_dropout": 0.2,
        "learning_rate": 5e-4, "batch_size": 64,
        "trainable_embeddings": True,
    },
    {
        "trial_name": "finetune_medium",
        "gru_units": 128, "dense_units": 64,
        "gru_dropout": 0.2, "dense_dropout": 0.4, "spatial_dropout": 0.2,
        "learning_rate": 5e-4, "batch_size": 64,
        "trainable_embeddings": True,
    },
    {
        "trial_name": "finetune_richer_head",
        "gru_units": 128, "dense_units": 128,
        "gru_dropout": 0.3, "dense_dropout": 0.5, "spatial_dropout": 0.3,
        "learning_rate": 3e-4, "batch_size": 32,
        "trainable_embeddings": True,
    },
]

pd.DataFrame(gru_search_space)

In [ ]:
# =============================================================================
# RUN HYPERPARAMETER TUNING — GRU
# Selection: best val_f1; tie-break on lowest best_val_loss
# =============================================================================
section("Run GRU hyperparameter tuning")

gru_tuning_rows = []
gru_best_config = None
gru_best_val_f1 = -1
gru_best_val_loss = np.inf

for i, config in enumerate(gru_search_space, start=1):
    print("\n" + "-" * 95)
    print(f"Trial {i}/{len(gru_search_space)} : {config['trial_name']}")
    print(config)

    _, _, summary, _ = train_one_gru(config=config, verbose=0)
    gru_tuning_rows.append(summary)

    print(f"Val accuracy : {summary['val_accuracy']:.4f}")
    print(f"Val F1       : {summary['val_f1']:.4f}")
    print(f"Val ROC-AUC  : {summary['val_roc_auc']:.4f}")
    print(f"Best val loss: {summary['best_val_loss']:.4f}")
    print(f"Epochs ran   : {summary['epochs_ran']}")

    if (summary["val_f1"] > gru_best_val_f1) or (
        np.isclose(summary["val_f1"], gru_best_val_f1)
        and summary["best_val_loss"] < gru_best_val_loss
    ):
        gru_best_val_f1   = summary["val_f1"]
        gru_best_val_loss = summary["best_val_loss"]
        gru_best_config   = config.copy()

gru_tuning_df = (
    pd.DataFrame(gru_tuning_rows)
    .sort_values(["val_f1", "val_accuracy", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

print("\nBest GRU configuration:")
print(gru_best_config)

print("\nValidation comparison table:")
print(
    gru_tuning_df[[
        "trial_name", "gru_units", "dense_units", "trainable_embeddings",
        "batch_size", "learning_rate", "epochs_ran",
        "val_accuracy", "val_f1", "val_roc_auc", "best_val_loss",
    ]].round(4).to_string(index=False)
)

In [ ]:
# =============================================================================
# VISUALISE TUNING RESULTS — GRU
# =============================================================================
section("GRU tuning comparison")

fig = px.bar(
    gru_tuning_df,
    x="trial_name", y="val_f1",
    color="trainable_embeddings",
    title="GRU tuning — validation macro F1 by trial",
    color_discrete_sequence=["#2E86AB", "#D1495B"],
    text="val_f1",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    title_x=0.02,
    yaxis=dict(title="Val macro F1", range=[0, 1]),
    xaxis_title=None,
    legend_title_text="Embeddings trainable",
)
fig.show()

In [ ]:
# =============================================================================
# FINAL TRAINING — GRU baseline vs tuned
# =============================================================================
section("Final GRU training: baseline vs tuned")

gru_baseline_config = gru_search_space[0]   # the reference config

gru_baseline_model, gru_baseline_history, gru_baseline_val_summary, gru_baseline_time = train_one_gru(
    config=gru_baseline_config, verbose=0,
)

gru_tuned_model, gru_tuned_history, gru_tuned_val_summary, gru_tuned_time = train_one_gru(
    config=gru_best_config, verbose=0,
)

plot_history(gru_baseline_history, "GRU (baseline)")
plot_history(gru_tuned_history,    "GRU (tuned)")

In [ ]:
# =============================================================================
# TEST SET EVALUATION — GRU (uses shared get_metrics helper)
# =============================================================================
section("GRU test evaluation")

gru_baseline_proba = gru_baseline_model.predict(X_test_seq, verbose=0)
gru_baseline_pred  = gru_baseline_proba.argmax(axis=1)

gru_tuned_proba = gru_tuned_model.predict(X_test_seq, verbose=0)
gru_tuned_pred  = gru_tuned_proba.argmax(axis=1)

gru_baseline_metrics = get_metrics(
    y_true=y_test, y_pred=gru_baseline_pred, y_proba=gru_baseline_proba,
    model_name="GRU (baseline)", tier="Deep Learning", time_s=gru_baseline_time,
)
gru_tuned_metrics = get_metrics(
    y_true=y_test, y_pred=gru_tuned_pred, y_proba=gru_tuned_proba,
    model_name="GRU (tuned)", tier="Deep Learning", time_s=gru_tuned_time,
)

results["GRU (baseline)"] = gru_baseline_metrics
results["GRU (tuned)"]    = gru_tuned_metrics

# Kasia - Bert Model

---
### Feature Engineering — Transformer BERT

Transformer models require text converted to **token IDs + attention masks** using the model's own pretrained tokeniser — not raw strings and not GloVe vectors.

**Why use the model's own tokeniser?**
DistilBERT and BERT were both pretrained with **WordPiece** subword tokenisation. Using a different tokeniser would mean every input token gets mapped to an unknown embedding and the pretrained knowledge would be destroyed. The tokeniser splits rare words into known subword pieces — e.g. "cancellations" → `["cancel", "##lations"]` — so the model rarely sees out-of-vocabulary tokens.

**Two checkpoints compared for this tier:**
- **DistilBERT** — 6 layers, 66M parameters, ~40% smaller and ~60% faster than BERT
- **BERT-base-uncased** — 12 layers, 110M parameters, full pretrained model

In [ ]:
# =============================================================================
# FEATURE ENGINEERING — Transformer: BERT
# Tokenisation + attention masks using the model's own pretrained tokeniser
# =============================================================================
section("Tokenisation — for NB3 Transformers")

from transformers import AutoTokenizer
from datasets import Dataset
import torch

MAX_LEN_TF = 64   # 95th percentile utterance length from EDA; transformers pad to this
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

def tokenise_split(texts, labels, tokenizer, max_len=MAX_LEN_TF):
    """
    Convert a list of texts + integer labels into a HuggingFace Dataset
    with input_ids, attention_mask, and label columns ready for Trainer.
    """
    ds = Dataset.from_dict({"text": list(texts), "label": list(labels)})
    ds = ds.map(
        lambda ex: tokenizer(ex["text"], truncation=True,
                             padding="max_length", max_length=max_len),
        batched=True
    )
    ds = ds.remove_columns(["text"])
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return ds

print(f"Max sequence length : {MAX_LEN_TF}")
print(f"Device              : {DEVICE}")
print("\n BERT — WordPiece subword tokenisation")


Tokenisation — for NB3 Transformers
Max sequence length : 64
Device              : cuda

 BERT — WordPiece subword tokenisation


### Why WordPiece Tokenisation?

DistilBERT and BERT cannot read raw strings — they need numbers. Each piece of
text is split into subword tokens, and each token is mapped to an integer ID
(e.g. "cancel" → 17195). These IDs are then passed through the pretrained
embedding layer of the transformer.

The simplest tokenisation would be to split on whitespace, but this would
produce many unknown tokens — any word not in the original vocabulary would
become `[UNK]` and lose all meaning. WordPiece avoids this by splitting rare
words into known subword pieces:

- "cancel" → `["cancel"]`
- "cancellations" → `["cancel", "##lations"]`
- "unsubscribing" → `["un", "##sub", "##scribing"]`

This guarantees almost every input is tokenised into meaningful pieces rather
than unknown tokens.

**Why not reuse the GloVe tokeniser?**
The transformer was pretrained with WordPiece. Its embedding layer expects
WordPiece token IDs — not GloVe indices. Using any other tokeniser would break
the pretrained model and every input would be treated as random noise.

**Two checkpoints are compared:**

| Checkpoint | Parameters | Layers | Why test both |
|------------|-----------|--------|---------------|
| DistilBERT | 66M | 6 | Faster — validates the full pipeline first |
| BERT-base  | 110M | 12 | Deeper — typically 1–2% higher F1 than DistilBERT |

DistilBERT is run first to confirm the training pipeline is correctly
configured. If the pipeline is broken, this is caught in ~20 minutes rather
than ~40 minutes with BERT.

### Build the BERT model

Below step is to define a reusable function that creates a transformer-based classifier. The model loads a pretrained checkpoint (DistilBERT or BERT), attaches a linear classification head on top of the `[CLS]` token, and returns the whole network ready for fine-tuning.

This structure is suitable for customer-support text because transformer attention allows every token to look at every other token, so the model can pick up intent cues no matter where they appear in the sentence.

In [109]:
# =============================================================================
# BERT MODEL BUILDER
# =============================================================================
section("Define BERT model")

from transformers import AutoModelForSequenceClassification

def build_bert_model(
    model_checkpoint="bert-base-uncased",
    num_labels=None,
    dropout=0.1,
    freeze_encoder=False
):
    """
    Build a transformer text classifier using the existing notebook variables.
    Works for any HuggingFace checkpoint exposing a sequence-classification head
    (e.g. bert-base-uncased, distilbert-base-uncased).
    """

    # Default to the notebook-level NUM_CLASSES constant if not overridden
    n_labels = num_labels if num_labels is not None else NUM_CLASSES

    # Load pretrained transformer + a fresh linear classification head
    # The head is a Linear(hidden_size, num_labels) layer on top of the [CLS] token
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_labels,
        hidden_dropout_prob=dropout,            # dropout inside the transformer
        attention_probs_dropout_prob=dropout,   # dropout on attention weights
        ignore_mismatched_sizes=True            # needed when swapping the head size
    )

    # Optionally freeze the pretrained encoder — only the classification head trains
    # Useful as a cheap baseline: tests whether pretrained features alone are enough
    if freeze_encoder:
        for name, param in model.named_parameters():
            if "classifier" not in name:
                param.requires_grad = False

    # Move model to the chosen device (GPU if available, else CPU)
    model.to(DEVICE)

    return model

print("build_bert_model() ready ")


Define BERT model
build_bert_model() ready 


### Why this BERT architecture was chosen

This architecture was selected because it is *appropriate for multi-class text classification.*

The pretrained transformer converts WordPiece token IDs into contextual embeddings — every token is represented by a vector that depends on every other token in the sentence. If DistilBERT or BERT weights are available (they always are via HuggingFace), the model starts with semantic knowledge learned from billions of words of Wikipedia + BooksCorpus. The parameter `freeze_encoder` allows the experiment to compare feature-extraction (frozen encoder) against full fine-tuning.

**Hidden-state dropout** (`hidden_dropout_prob`) and **attention dropout** (`attention_probs_dropout_prob`) are used to improve generalisation by dropping parts of the hidden representations and attention weights during training.

The self-attention mechanism reads the message in all directions at once. This is more expressive than BiLSTM because every token can directly attend to every other token, instead of passing information through a sequential hidden state.

**The `[CLS]` token pooling** replaces `GlobalMaxPooling1D`. During pretraining BERT was trained to use the `[CLS]` token as an aggregate sentence representation, so its final hidden state is naturally suited for classification.

The output layer uses softmax because this is a multi-class classification problem with `NUM_CLASSES` intent labels.

The loss function is sparse categorical cross-entropy — computed internally by HuggingFace `AutoModelForSequenceClassification` when integer `label` columns are supplied.

### Training helper for BERT experiments

The following function trains one BERT configuration and returns:

1. The trained model
2. The training history (per-epoch val loss and val F1)
3. A validation summary used for hyperparameter tuning
4. The total training time

The validation metrics are computed directly inside the function so that the shared helper `get_metrics()` can remain the common final evaluation function for all models.

In [ ]:
# =============================================================================
# TRAINING HELPER FOR BERT
# =============================================================================
section("BERT training helper")

from transformers import (TrainingArguments, Trainer,
                          EarlyStoppingCallback)

def _compute_metrics_hf(eval_pred):
    """Macro F1 + accuracy — used by Trainer after every validation epoch."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro":  f1_score(labels, preds, average="macro"),
        "accuracy":  accuracy_score(labels, preds)
    }

def train_one_bert(config, verbose=0):
    """
    Train one BERT / DistilBERT configuration and evaluate it on the validation set.
    Returns model, history, validation summary, and training time.
    """
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # Load the pretrained tokeniser for this checkpoint
    tokenizer = AutoTokenizer.from_pretrained(config["model_checkpoint"])

    # Tokenise all three splits with this tokeniser
    train_ds = tokenise_split(X_train, y_train, tokenizer)
    val_ds   = tokenise_split(X_val,   y_val,   tokenizer)
    test_ds  = tokenise_split(X_test,  y_test,  tokenizer)

    # Build the model for this trial
    model = build_bert_model(
        model_checkpoint=config["model_checkpoint"],
        num_labels=NUM_CLASSES,
        dropout=config["dropout"],
        freeze_encoder=config["freeze_encoder"]
    )

    # Trainer arguments — warm-up schedule, gradient clipping, early stopping
    args = TrainingArguments(
        output_dir=f"./bert_out/{config['trial_name']}",
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"] * 2,
        learning_rate=config["learning_rate"],
        warmup_ratio=config["warmup_ratio"],   # LR warmup (10% of steps)
        weight_decay=0.01,
        max_grad_norm=1.0,                     # gradient clipping
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        fp16=(DEVICE == "cuda"),               # mixed precision on GPU only
        report_to="none",
        disable_tqdm=(verbose == 0)
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=_compute_metrics_hf,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    start = time.time()
    trainer.train()
    train_time = time.time() - start

    # Collect per-epoch history from Trainer state — mirrors Keras history
    history = {"loss": [], "val_loss": [], "val_f1": []}
    for log in trainer.state.log_history:
        if "loss" in log and "epoch" in log and "eval_loss" not in log:
            history["loss"].append(log["loss"])
        if "eval_loss" in log:
            history["val_loss"].append(log["eval_loss"])
            history["val_f1"].append(log.get("eval_f1_macro", np.nan))

    # Validation predictions for tuning
    val_logits = trainer.predict(val_ds).predictions
    val_proba  = softmax(val_logits, axis=1)
    val_pred   = val_proba.argmax(axis=1)

    val_summary = {
        "trial_name":       config["trial_name"],
        "model_checkpoint": config["model_checkpoint"],
        "learning_rate":    config["learning_rate"],
        "batch_size":       config["batch_size"],
        "dropout":          config["dropout"],
        "warmup_ratio":     config["warmup_ratio"],
        "freeze_encoder":   config["freeze_encoder"],
        "epochs_ran":       len(history["val_loss"]),
        "best_val_loss":    float(np.min(history["val_loss"])) if history["val_loss"] else np.nan,
        "train_time_s":     round(train_time, 1),
        "val_accuracy":     accuracy_score(y_val, val_pred),
        "val_precision":    precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall":       recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1":           f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc":      roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
    }

    return model, history, val_summary, train_time

print("train_one_bert() ready ")


### Why this training function is needed

This helper function makes hyperparameter tuning easier and keeps the notebook organised.

**EarlyStoppingCallback** is used to stop training when the validation F1 stops improving. This reduces overfitting and avoids wasting training time — transformers converge quickly (often within 3 epochs) and continuing past the best epoch only degrades test performance.

**Warm-up ratio** ramps the learning rate up gradually over the first 10% of steps. This is critical for transformers — jumping straight to the maximum learning rate at step 1 destroys pretrained weights before the model has adapted to the new task.

**Gradient clipping** at `max_grad_norm=1.0` prevents exploding gradients. Transformers are sensitive to large updates during the early warm-up phase; clipping keeps training stable.

**Mixed precision (fp16)** is enabled automatically when a GPU is available. It makes training ~2× faster with negligible accuracy loss and is essential for BERT-base on limited hardware.

The model is tuned using validation **macro F1** because this is a multi-class intent classification task. Macro F1 is more informative than accuracy alone because it gives equal importance to each class.

## Define the hyperparameter search space

The next cell defines a small but meaningful tuning space for BERT. The selected hyperparameters control the pretrained backbone, regularisation strength, optimisation behaviour, and whether the pretrained encoder is frozen or fine-tuned.

### Why these hyperparameters were selected

The tuning space focuses on the parameters most likely to influence transformer performance.

***model_checkpoint***: compares DistilBERT (cheap) with BERT-base (expensive)

***learning_rate***: the most important transformer hyperparameter — values outside 1e-5 to 5e-5 typically fail

***dropout***: hidden + attention dropout provide regularisation

***warmup_ratio***: affects optimisation stability early in training

***batch_size***: affects both speed and gradient behaviour (smaller = noisier but often generalises better)

***freeze_encoder***: compares feature-extraction against full fine-tuning

This search space is intentionally small (4 trials) because each transformer run is expensive. The BiLSTM search space used 6 trials because each trial takes ~1 minute; each BERT trial takes 10–40 minutes depending on hardware.

In [ ]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE
# =============================================================================
section("BERT hyperparameter search space")

bert_search_space = [
    {
        "trial_name":       "distilbert_baseline",
        "model_checkpoint": "distilbert-base-uncased",
        "learning_rate":    2e-5,
        "batch_size":       32,
        "dropout":          0.1,
        "warmup_ratio":     0.1,
        "epochs":           4,
        "freeze_encoder":   False
    },
    {
        "trial_name":       "distilbert_higher_lr",
        "model_checkpoint": "distilbert-base-uncased",
        "learning_rate":    3e-5,
        "batch_size":       32,
        "dropout":          0.1,
        "warmup_ratio":     0.1,
        "epochs":           4,
        "freeze_encoder":   False
    },
    {
        "trial_name":       "bert_baseline",
        "model_checkpoint": "bert-base-uncased",
        "learning_rate":    2e-5,
        "batch_size":       16,
        "dropout":          0.1,
        "warmup_ratio":     0.1,
        "epochs":           4,
        "freeze_encoder":   False
    },
    {
        "trial_name":       "bert_higher_dropout",
        "model_checkpoint": "bert-base-uncased",
        "learning_rate":    2e-5,
        "batch_size":       16,
        "dropout":          0.2,
        "warmup_ratio":     0.1,
        "epochs":           4,
        "freeze_encoder":   False
    }
]

pd.DataFrame(bert_search_space)


# Models Evaluation

Every model is evaluated on the held-out test set using the same set of metrics:

- **Accuracy** — overall proportion of correct predictions.
- **Macro precision, recall, F1** — each class contributes equally to the score, so rare classes are not hidden by common ones.
- **ROC-AUC (macro, one-vs-rest)** — measures how well the model ranks the correct class relative to the other 26, averaged across all classes.

Macro F1 is the main metric for model selection because it gives equal weight to every intent, which matches the business view that all 27 intents are equally important to classify correctly.

The following visualisations support the evaluation:

- **Training / validation loss curves** — to check that each model converged and did not overfit.
- **Confusion matrices** — to see which intent pairs are most often confused with each other.
- **Per-intent F1 scores** — to identify which classes are easy and which are hard for each model.
- **Attention maps (transformers only)** — to show which words the transformer focused on when making its prediction.

LinearSVC has no native probability output, so its `decision_function` scores are passed through softmax inside the shared evaluation helper before ROC-AUC is computed. This is the only place model-specific evaluation logic is needed.

## Cross-Model Consolidation

All trained models are gathered into a single comparison structure here. This avoids renaming variables across the notebook and gives one place to:

1. compare metrics side-by-side
2. plot a unified comparison chart
3. plot confusion matrices for every model

In [ ]:
# =============================================================================
# CROSS-MODEL CONSOLIDATION — registry + master table
# =============================================================================
section("Cross-model consolidation")

model_registry = {
    "Logistic Regression": {
        "y_pred":  globals().get("y_pred_lr"),
        "y_proba": globals().get("y_proba_lr"),
        "metrics": globals().get("lr_metrics"),
        "tier":    "Classical ML",
    },
    "LinearSVC": {
        "y_pred":  globals().get("y_pred_svc"),
        "y_proba": globals().get("y_proba_svc"),
        "metrics": globals().get("svc_metrics"),
        "tier":    "Classical ML",
    },
    "BiLSTM (baseline)": {
        "y_pred":  globals().get("baseline_pred"),
        "y_proba": globals().get("baseline_proba"),
        "metrics": globals().get("baseline_metrics"),
        "tier":    "Deep Learning",
    },
    "BiLSTM (tuned)": {
        "y_pred":  globals().get("tuned_pred"),
        "y_proba": globals().get("tuned_proba"),
        "metrics": globals().get("tuned_metrics"),
        "tier":    "Deep Learning",
    },
    # Add GRU / BERT entries as you build them
}

trained = {k: v for k, v in model_registry.items() if v["y_pred"] is not None}
print(f"Models in comparison: {len(trained)}")
for m in trained:
    print(f"  ✓ {m}")

rows = []
for name, info in trained.items():
    m = info["metrics"] or {}
    rows.append({
        "Model":     name,
        "Tier":      info["tier"],
        "Accuracy":  m.get("accuracy"),
        "Precision": m.get("precision"),
        "Recall":    m.get("recall"),
        "F1":        m.get("f1"),
        "ROC-AUC":   m.get("roc_auc"),
        "Time (s)":  m.get("time_s") or m.get("Time(s)"),
    })

master_df = (
    pd.DataFrame(rows)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\n" + master_df.round(4).to_string(index=False))
master_df.to_csv("nb_all_models_results.csv", index=False)
print("\nSaved → nb_all_models_results.csv  (used by NB4)")

master_df.style \
    .background_gradient(subset=["F1"],        cmap="Blues") \
    .background_gradient(subset=["ROC-AUC"],   cmap="Greens") \
    .background_gradient(subset=["Precision"], cmap="Oranges") \
    .format({c: "{:.4f}" for c in ["Accuracy","Precision","Recall","F1","ROC-AUC"]}) \
    .format({"Time (s)": "{:.1f}"})

In [ ]:
# =============================================================================
# COMPARISON CHART — grouped bar of all key metrics
# =============================================================================
section("Comparison chart — all models")

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
plot_df = master_df.melt(
    id_vars=["Model", "Tier"],
    value_vars=metric_cols,
    var_name="Metric",
    value_name="Score",
)

fig = px.bar(
    plot_df,
    x="Model", y="Score", color="Metric",
    barmode="group",
    title="Cross-tier model comparison — all metrics",
    color_discrete_sequence=["#2E86AB", "#D1495B", "#F4A261", "#2A9D8F", "#6A4C93"],
)
fig.update_layout(
    title_x=0.02,
    yaxis=dict(title="Score", range=[0, 1.0]),
    xaxis_title=None,
    legend_title_text="Metric",
    height=500,
)
fig.show()

In [ ]:
# =============================================================================
# F1 BY TIER — quick view of best model per family
# =============================================================================
section("Best F1 per tier")

best_per_tier = (
    master_df.sort_values("F1", ascending=False)
    .groupby("Tier", as_index=False).first()
)

fig = px.bar(
    best_per_tier.sort_values("F1"),
    x="F1", y="Tier", color="Model",
    orientation="h",
    title="Best macro F1 per tier",
    text="Model",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(title_x=0.02, xaxis=dict(range=[0, 1]), height=350,
                  showlegend=False)
fig.update_traces(textposition="inside")
fig.show()

In [ ]:
# =============================================================================
# 5) CONFUSION MATRICES — one per model, on a shared scale
# =============================================================================
section("Confusion matrices — every trained model")

from plotly.subplots import make_subplots

n = len(trained)
cols = 2
rows = (n + cols - 1) // cols

fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=list(trained.keys()),
    horizontal_spacing=0.18, vertical_spacing=0.12,
)

for idx, (name, info) in enumerate(trained.items()):
    r, c = idx // cols + 1, idx % cols + 1
    cm   = confusion_matrix(y_test, info["y_pred"])
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=le.classes_, y=le.classes_,
            colorscale="Blues",
            showscale=(idx == 0),
            hovertemplate="True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>",
        ),
        row=r, col=c,
    )
    fig.update_xaxes(tickangle=45, tickfont=dict(size=8), row=r, col=c)
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=8), row=r, col=c)

fig.update_layout(
    title="Confusion matrices — all models (test set)",
    title_x=0.02,
    height=420 * rows,
    margin=dict(t=80, l=40, r=40, b=40),
)
fig.show()